# 19.23 新版判斷器怎樣共用材料並處理候選變化？


本節保留C-v2的共享計算設計與當時未達標的實測。後續合格的裝置成品是[19.25](https://birdhackor.github.io/tiny-perceptron-vlm/19.25.html)的C-v3；本節的byte介面、權重與採用規則不是C-v3的替換配套。

材料是「使用者要查卡片進度，申請編號K17」。現在問服務、資訊是否齊備，再按規準分級；三題都需要讀相同材料。舊[19.16](https://birdhackor.github.io/tiny-perceptron-vlm/19.16.html)把材料和每個候選重新編碼；新版C要沿[16.15](https://birdhackor.github.io/tiny-perceptron-vlm/16.15.html)的分工，先算一次材料記憶體M，再讓各題各候選唯讀它。前置還包括[7.26的候選變化標籤](https://birdhackor.github.io/tiny-perceptron-vlm/7.26.html)、[9.17的評分](https://birdhackor.github.io/tiny-perceptron-vlm/9.17.html)及[9.18的校準和接受規則](https://birdhackor.github.io/tiny-perceptron-vlm/9.18.html)。

新入口直接收文字`state`、問題型別`kind`、問題文字及候選描述，**不經舊C或B的`parse_state`**。文字採完整UTF-8 bytes：0–255都有ID，PAD與特殊符號另設ID。state最多512 bytes，問題與每項描述各128 bytes；例如171個「漢」有513 bytes，不能當成171 bytes而放行。這避免遇到陌生字便缺字，沒有保證理解陌生說法。

Choice傳1–7個實質候選，模型最後另加none。info固定yes／no；rubric固定0／1／2，問題與各級描述合起來須給本題可核對的分級規準。規準可以像後面的正式例一樣集中寫在問題，每級描述仍須能辨認對應的等級。候選ID只追蹤身分，不當作語義輸入；描述連同本題規則須能辨認該候選，不能指「上一項」。ID與完整描述不得重複。程式可拒絕相同字串，語義重複及跨候選指涉仍需資料稽核；不能靠不同ID把兩個同義正解硬拆開。這個介面可寫客服例子，格式合法卻不代表已受訓支援：本輪正式C-v2核心只在後文的三裝置規則世界學習、驗收，不能把byte覆蓋當成通用客服或算式能力。

以下是完整的小型CPU參考實作，權重只隨機初始化，**沒有訓練、沒有語義正確率或新版checkpoint**。每題單獨運算，所以沒有補齊的材料／候選；平均、最大值只取有效項。`branches`逐項用同一套參數，實質候選完全不讀彼此；none才讀問題、候選特徵的平均、最大值及數量。這讓none知道整份列表，同時保住重排關係。

選答案先比較原始`scores`。電腦只能保存有限精度，例如float32的兩個分數`[0,1e-8]`在T=1做softmax，儲存的機率都會成為0.5，原分數仍有大小。因此`selected`先選原分數最大者，原分數真的相等時才按固定ID決定。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import math
import torch
from torch import nn

PAD, BOS, SEP = 256, 257, 258
TYPE = {"choice": 259, "info": 260, "rubric": 261}


def utf8(text, limit=None):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("必須是非空文字")
    try:
        raw = text.encode("utf-8", errors="strict")
    except UnicodeEncodeError:
        raise ValueError("非法Unicode") from None
    if limit is not None and len(raw) > limit:
        raise ValueError("超過bytes上限")
    return list(raw)


def validate(kind, question, candidates):
    if not isinstance(kind, str) or kind not in TYPE:
        raise ValueError("未知問題型別")
    utf8(question, 128)
    if not isinstance(candidates, (list, tuple)) or not candidates:
        raise ValueError("候選不可空")
    ids, descriptions = [], []
    for pair in candidates:
        if not isinstance(pair, (list, tuple)) or len(pair) != 2:
            raise ValueError("每項需是(ID, 描述)")
        cid, description = pair
        utf8(cid)  # ID只驗合法字串，不送進模型
        utf8(description, 128)
        ids.append(cid)
        descriptions.append(description)
    if len(set(ids)) != len(ids) or len(set(descriptions)) != len(descriptions):
        raise ValueError("ID或描述重複")
    legal = (
        (1 <= len(ids) <= 7 and "none" not in ids)
        if kind == "choice"
        else (set(ids) == ({"yes", "no"} if kind == "info" else {"0", "1", "2"}))
    )
    if not legal:
        raise ValueError("候選不符合型別")


class TinyDecision(nn.Module):
    def __init__(self, width=16):
        super().__init__()
        self.width = width
        self.byte = nn.Embedding(262, width, padding_idx=PAD)
        self.position = nn.Embedding(513, width)
        self.sq, self.sk, self.sv = [nn.Linear(width, width) for _ in range(3)]
        self.token = nn.Sequential(nn.Linear(width, width), nn.Tanh())
        self.cq, self.ck, self.cv = [nn.Linear(width, width) for _ in range(3)]
        self.score_q, self.score_k = [nn.Linear(width, width) for _ in range(2)]
        self.none = nn.Linear(3 * width + 1, 1)

    def vectors(self, ids):
        device = self.byte.weight.device
        ids = torch.tensor(ids, device=device, dtype=torch.long)
        positions = torch.arange(len(ids), device=device)
        return self.byte(ids) + self.position(positions)

    def pool(self, ids):
        return self.token(self.vectors(ids)).mean(0)

    def encode_state(self, state):
        x = self.vectors([BOS] + utf8(state, 512))
        attention = (self.sq(x) @ self.sk(x).T / math.sqrt(self.width)).softmax(-1)
        return torch.tanh(x + attention @ self.sv(x))  # 只讀state

    def score(self, memory, kind, question, candidates, T=1.0):
        validate(kind, question, candidates)
        if not math.isfinite(T) or T <= 0:
            raise ValueError("T須為正有限數")
        question_ids = [TYPE[kind]] + utf8(question, 128)
        q = self.pool(question_ids)
        queries, weights, hidden = [], [], []
        # 這兩份材料投影可供本題全部候選唯讀。
        keys, values = self.ck(memory), self.cv(memory)
        for _, description in candidates:
            u = self.pool(question_ids + [SEP] + utf8(description, 128))
            query = self.cq(u)
            a = (query @ keys.T / math.sqrt(self.width)).softmax(-1)
            h = torch.tanh(u + a @ values)
            queries.append(query)
            weights.append(a)
            hidden.append(h)
        H = torch.stack(hidden)
        scores = self.score_k(H) @ self.score_q(q) / math.sqrt(self.width)
        ids = [cid for cid, _ in candidates]
        if kind == "choice":
            summary = torch.cat([q, H.mean(0), H.max(0).values, H.new_tensor([len(H)])])
            scores = torch.cat([scores, self.none(summary)])
            ids = ids + ["none"]
        return {
            "ids": ids,
            "query": torch.stack(queries),
            "attention": torch.stack(weights),
            "hidden": H,
            "scores": scores,
            "p": (scores / T).softmax(-1),
        }

    def forward(self, state, kind, question, candidates, T=1.0):
        memory = self.encode_state(state)
        return memory, self.score(memory, kind, question, candidates, T)


def selected(result):
    # softmax捨入可能讓不同分數顯示同機率，先按原始分數取最大。
    # 原分數真正平手時才按固定ID；不可用清單位置挑第一個。
    i = min(range(len(result["ids"])), key=lambda j: (-float(result["scores"][j]), result["ids"][j]))
    return result["ids"][i]


torch.manual_seed(7)
torch.set_num_threads(1)
model = TinyDecision().float().eval()
state = "使用者要查卡片進度，申請編號K17。"
choice = ("choice", "使用者需要哪個服務？", [("card", "查卡片申請進度"), ("address", "查地址變更規則")])
info = ("info", "查卡片所需的申請編號是否已提供？", [("yes", "已提供申請編號"), ("no", "未提供申請編號")])
rubric = ("rubric", "評任務與申請編號兩欄的完整度。", [("0", "兩欄都缺"), ("1", "只有一欄"), ("2", "兩欄都有")])

with torch.no_grad():
    M = model.encode_state(state)
    snapshot = M.clone()
    shared = [model.score(M, *q) for q in [choice, info, rubric, choice]]
    error = 0.0
    for q, reused in zip([choice, info, rubric, choice], shared):
        fresh_M, fresh = model(state, *q)
        error = max(error, float((M - fresh_M).abs().max()))
        for field in ["query", "attention", "hidden", "scores", "p"]:
            error = max(error, float((fresh[field] - reused[field]).abs().max()))
    reverse = model.score(M, choice[0], choice[1], list(reversed(choice[2])))
    before = dict(zip(shared[0]["ids"], shared[0]["p"].tolist()))
    after = dict(zip(reverse["ids"], reverse["p"].tolist()))
    reorder_error = max(abs(before[cid] - after[cid]) for cid in before)
    print("三型候選數", [len(r["ids"]) for r in shared[:3]])
    print("重算／共享最大差", error, "重排最大差", reorder_error)
    print("Q1重問機率差", float((shared[0]["p"] - shared[-1]["p"]).abs().max()))
    print("M保持原值", torch.equal(M, snapshot))
    assert error <= 1e-5 and reorder_error <= 1e-5 and torch.equal(M, snapshot)
    orders = [["card", "address"], ["address", "card"]]
    print("人工平手按ID", [selected({"ids": order, "scores": torch.tensor([0.0, 0.0])}) for order in orders])

bad_inputs = [
    ("超長state", lambda: model.encode_state("漢" * 171)),
    ("非法Unicode", lambda: model.encode_state("x\ud800")),
    ("空問題", lambda: model.score(M, "choice", "", choice[2])),
    ("重複描述", lambda: model.score(M, "choice", "選服務", [("a", "查卡片"), ("b", "查卡片")])),
    ("非法info候選", lambda: model.score(M, "info", "資訊是否齊？", [("yes", "齊"), ("maybe", "可能齊")])),
]
for name, call in bad_inputs:
    try:
        call()
    except ValueError as error:
        print("拒絕", name, str(error))

程式裡，`q`是問題側的特徵，`u`把問題與當前候選描述一起編成分支特徵，`H`再把各分支讀取M後的結果疊成一列一個候選的表。各實質候選的分數逐列計算；none才讀整張表的彙整。

關鍵輸出如下：

```text
三型候選數 [3, 2, 3]
重算／共享最大差 0.0 重排最大差 0.0
Q1重問機率差 0.0
M保持原值 True
人工平手按ID ['address', 'address']
拒絕 超長state 超過bytes上限
拒絕 非法Unicode 非法Unicode
拒絕 空問題 必須是非空文字
拒絕 重複描述 ID或描述重複
拒絕 非法info候選 候選不符合型別
```

這些核對計算關係，沒有檢查哪個候選意義正確。非法輸入會明確拒絕，不截斷、替換字元或偷改候選；這是格式錯誤，和合法輸入落在支援世界外不同。正式快取還要按16.15綁材料bytes、權重及輸入處理版本；更新權重便不能沿用舊M。

訓練時同題全部`scores`一起算交叉熵；gold位置由`result["ids"].index(gold_id)`求出，候選重排就跟著重新找位置。Choice的none正解也需7.26那種配對示範；對稱摘要不會自動教會「都不適合」。這個CPU程式沒有更新參數。

訓練資料按可見情境建立家族，先分train、model-dev、calibration、policy、final，再產生同家族的問題、缺欄與候選變體；還要查完整文字與實際token觀測跨份重複，不能只相信隨機ID。上限是訓練30000、model-dev 2000家族，另留校準80、決策60及核心final 60家族。以下核心直接示出**等家族、等型別**的溫度選擇；每家族各有choice、info、rubric一個代表題，每題傳原始分數與gold位置。


In [ ]:
def fit_temperature(calibration):
    grid = [0.25 * 2 ** (k / 8) for k in range(33)]

    def nll(T):
        family_losses = []
        for family in calibration:
            typed = []
            for kind in ["choice", "info", "rubric"]:
                raw_scores, gold = family[kind]
                typed.append(float(-torch.log_softmax(raw_scores / T, -1)[gold]))
            family_losses.append(sum(typed) / 3)
        return sum(family_losses) / len(family_losses)

    return min(grid, key=lambda T: (nll(T), abs(T - 1), T))


# 純數學核對：全部分數相等時，任意T的代價相同，照平手規則選1。
fixture = [{kind: (torch.zeros(n), 0) for kind, n in [("choice", 3), ("info", 2), ("rubric", 3)]}]
print("等分數人工校準T", fit_temperature(fixture))

這個人工核對印出`等分數人工校準T 1.0`，只檢查計算與平手規則，沒有替新版模型選好溫度。

先固定T，再照9.18只在policy選τ。接受只看模型的`p_max`、非none及資訊充分預測yes；評分時，Choice錯、gold不支援或gold資訊不足任一成立都是誤收。覆蓋率分母始終含全部60個預定Choice，零接受的risk未定義。主要目標是至少接30題、接受錯誤率不超過5%；它是有限樣本的事前目標，沒有未來5%保證。

核心final每家族三型各一題，所以每型n=60，不能加成180個獨立家族。Choice事前分48個支援且齊備、6個正解不在列表、6個支援世界內缺必要欄位；門檻是Choice至少54/60，info與rubric各至少51/60。`none`只說列表沒有合適項；未知任務的`unsupported`另外在獨立Auto 100家族查，不能給沒有定義的資訊問題硬貼no或grade 0。不同語言、來源、候選數與失敗原因另列，不能用平均分掩蓋。

練習加入第三個實質候選`("bill", "查帳單明細")`，比較原候選的`scores`和`p`：獨立分支分數應保持，softmax分母及none摘要卻可改。請解釋為什麼機率改變不等於共享材料遭到污染。

正式模型位於`tiny_perceptron/omni_products/choice.py`，完整計算另列於本節後段，實際密封測試與成績須另看驗收紀錄。本節展示新版計算契約，不替換19.16的歷史權重與舊結果；新tokenizer、架構和資料一起改，也不能把產品差異單獨歸功於pointer。

來源：本課新版範圍依[本輪具體契約](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-revision-20261029-omni/CONTRACTS.md#c的新介面與統計)；外部機制對照為[Kev固定版候選點積及prefix分支](https://github.com/jaredpalmer/kev/blob/2074cc44256200bb37540103fa4572901faa10f3/kev/model.py#L205-L461)、[固定版校準程式](https://github.com/jaredpalmer/kev/blob/2074cc44256200bb37540103fa4572901faa10f3/kev/metrics.py#L233-L295)。這裡的byte入口、state-only編碼器和對稱none是本課原創參考實作，沒有載入Kev或Qwen權重。

**正式C-v2沿用哪些計算？** 正式模型在 `tiny_perceptron/omni_products/choice.py`，不是把舊C權重改名。它與上面的短例都把材料和問題分開、逐候選讀同一份材料、以對稱摘要算none；實際容量和分支內容如下。

| 部分 | 正式計算 |
| --- | --- |
| 材料 | UTF-8 bytes加CLS；64維、4 heads、2層雙向encoder；只讀state |
| 每個分支 | `[CLS, question bytes, SEP, description bytes]`；共用一層64維encoder；不接別項候選 |
| 讀材料 | 取分支CLS，加問題型別向量，經兩層cross-attention唯讀M |
| 實質分數 | 同一個MLP逐分支計分；候選ID只用來追蹤及決平手 |
| none | 有效候選hidden的mean/max、分數mean/max與count/7；不把padding算入 |
| 快取 | state bytes、encoder參數、前處理、位置、dtype/device的hash；回傳副本防止呼叫端寫壞內部M |

分支CLS會先讀本題的問題與描述，才去讀材料；因此它代表這個候選在這個問題下需要查的線索。資料不夠時，架構本身不會自動推導none或資訊不足，仍需要帶標籤的示範。

<details>
<summary>正式C-v2的文字輸入驗證、編碼、共享材料與候選計算</summary>

下列程式直接內建正式實作，含batch補齊、none位置與cache失效條件。閱讀時先對上表中的 `state_encoder`、`branch_encoder`、`readers`、`score`、`none`五個零件，再看 `collate_questions`怎樣把多題各候選分成batch的獨立列。推論不需要gold；只有訓練時的labels才從gold ID找索引。

直接呼叫這裡的`ChoiceModel.predict`時，呼叫端須先確保`temperature`為正有限值；下例沿用預設1.0。這個method只檢查`temperature <= 0`，會漏過NaN與正無窮；正式服務的`CJudge`與校準入口另用`math.isfinite`拒絕非有限值，本輪固定T也通過該檢查。


In [ ]:
import hashlib
import json
from dataclasses import dataclass

import torch
import torch.nn.functional as F
from torch import nn

PAD, CLS, SEP = 256, 257, 258
PREPROCESSING = "strict-utf8-bytes-0-255-right-pad256-v1"
TYPES = {"choice": 0, "info": 1, "rubric": 2}


def text_bytes(text, limit, name):
    if type(text) is not str or not text.strip():
        raise ValueError(f"{name} 必須是非空文字")
    try:
        raw = text.encode("utf-8", errors="strict")
    except UnicodeError as error:
        raise ValueError(f"{name} 含非法 Unicode surrogate") from error
    if len(raw) > limit:
        raise ValueError(f"{name} 超過 {limit} bytes")
    return raw


@dataclass(frozen=True)
class Candidate:
    id: str
    description: str


@dataclass(frozen=True)
class Question:
    state: str
    kind: str
    text: str
    candidates: tuple[Candidate, ...]
    gold: str | None = None

    def validate(self):
        text_bytes(self.state, 512, "state")
        text_bytes(self.text, 128, "question")
        if self.kind not in TYPES:
            raise ValueError("僅支援 choice/info/rubric")
        if not 1 <= len(self.candidates) <= 7:
            raise ValueError("實質候選 1–7，加 none 最多 8")
        ids, descriptions = [], []
        for candidate in self.candidates:
            text_bytes(candidate.id, 64, "stable ID")
            text_bytes(candidate.description, 128, "candidate description")
            if candidate.id == "none":
                raise ValueError("none 為系統候選，不能當成實質候選")
            ids.append(candidate.id)
            descriptions.append(candidate.description)
        if len(set(ids)) != len(ids) or len(set(descriptions)) != len(descriptions):
            raise ValueError("候選 ID／完整描述不得重複；語義等價項也須由資料作者先去重")
        if self.kind == "info" and set(ids) != {"yes", "no"}:
            raise ValueError("info 固定 yes/no")
        if self.kind == "rubric" and set(ids) != {"0", "1", "2"}:
            raise ValueError("rubric 固定 0/1/2")
        if self.gold is not None and self.gold not in {*ids, "none"}:
            raise ValueError("gold 不在實質候選／none")
        if self.kind != "choice" and self.gold == "none":
            raise ValueError("info/rubric 不另加 none 類別")
        return self


def pad_bytes(rows, device):
    width = max(map(len, rows))
    tokens = torch.full((len(rows), width), PAD, dtype=torch.long, device=device)
    for index, row in enumerate(rows):
        tokens[index, : len(row)] = torch.tensor(row, device=device)
    return tokens


def collate_questions(questions, device="cpu"):
    questions = [question.validate() for question in questions]
    count = max(len(question.candidates) for question in questions)
    states, branches, kinds, counts, labels = [], [], [], [], []
    for question in questions:
        states.append([CLS, *text_bytes(question.state, 512, "state")])
        q = list(text_bytes(question.text, 128, "question"))
        kinds.append(TYPES[question.kind])
        counts.append(len(question.candidates))
        ids = [candidate.id for candidate in question.candidates] + (["none"] if question.kind == "choice" else [])
        labels.append(ids.index(question.gold) if question.gold is not None else -100)
        for i in range(count):
            description = question.candidates[i].description if i < len(question.candidates) else "padding"
            branches.append([CLS, *q, SEP, *text_bytes(description, 128, "description")])
    return {
        "state": pad_bytes(states, device),
        "branches": pad_bytes(branches, device),
        "kinds": torch.tensor(kinds, device=device),
        "counts": torch.tensor(counts, device=device),
        "labels": torch.tensor(labels, device=device),
        "max_candidates": count,
    }


@dataclass(frozen=True)
class ChoiceConfig:
    width: int = 64
    heads: int = 4
    state_layers: int = 2
    branch_layers: int = 1
    dropout: float = 0.0
    state_limit: int = 512
    question_limit: int = 128
    candidate_limit: int = 128


@dataclass(frozen=True)
class StateMemory:
    key: str
    values: torch.Tensor
    padding: torch.Tensor


class ByteEncoder(nn.Module):
    def __init__(self, config, length, layers):
        super().__init__()
        self.bytes = nn.Embedding(259, config.width, padding_idx=PAD)
        self.positions = nn.Embedding(length, config.width)
        layer = nn.TransformerEncoderLayer(
            config.width, config.heads, config.width * 4, dropout=0.0, batch_first=True, norm_first=True
        )
        self.layers = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(config.width)

    def forward(self, tokens):
        length = tokens.shape[1]
        embedded = self.bytes(tokens) + self.positions(torch.arange(length, device=tokens.device))
        return self.norm(self.layers(embedded, src_key_padding_mask=tokens.eq(PAD)))


class ChoiceModel(nn.Module):
    def __init__(self, config=ChoiceConfig()):
        super().__init__()
        if (
            config.dropout != 0
            or config.state_limit != 512
            or config.question_limit != 128
            or config.candidate_limit != 128
        ):
            raise ValueError("C-v2 固定 byte limits 且 dropout=0")
        self.config = config
        width = config.width
        self.state_encoder = ByteEncoder(config, 513, config.state_layers)
        self.branch_encoder = ByteEncoder(config, 258, config.branch_layers)
        self.types = nn.Embedding(3, width)
        self.readers = nn.ModuleList(
            nn.MultiheadAttention(width, config.heads, dropout=0.0, batch_first=True) for _ in range(2)
        )
        self.reader_norms = nn.ModuleList(nn.LayerNorm(width) for _ in range(2))
        self.score = nn.Sequential(nn.Linear(width, width), nn.GELU(), nn.Linear(width, 1))
        self.none = nn.Sequential(nn.Linear(width * 2 + 3, width), nn.GELU(), nn.Linear(width, 1))
        self._cache = {}

    def encode_state(self, state):
        return self.state_encoder(state)

    def forward(self, batch, recompute=False, memory=None):
        states, branch_tokens = batch["state"], batch["branches"]
        b, candidates = len(states), batch["max_candidates"]
        branches = (
            self.branch_encoder(branch_tokens)[:, :1]
            + self.types(batch["kinds"].repeat_interleave(candidates))[:, None]
        )
        if memory is not None:
            if self.training or recompute or b != 1:
                raise ValueError("外部 state cache 僅供單題 frozen eval")
            state_text = bytes(states[0, 1:][states[0, 1:] != PAD].tolist()).decode("utf-8", errors="strict")
            if memory.key != self.memory_key(state_text):
                raise ValueError("state cache 身分與本題／encoder 不符")
            state_values, padding = memory.values, memory.padding
            expanded = state_values.expand(candidates, -1, -1)
            pad = padding.expand(candidates, -1)
        elif recompute:
            # 真正重算 state-only encoder；不是 clone 已算 M 的假控制。
            expanded = self.encode_state(states.repeat_interleave(candidates, dim=0))
            pad = states.eq(PAD).repeat_interleave(candidates, dim=0)
        else:
            expanded = self.encode_state(states).repeat_interleave(candidates, dim=0)
            pad = states.eq(PAD).repeat_interleave(candidates, dim=0)
        for reader, norm in zip(self.readers, self.reader_norms):
            branches = norm(
                branches + reader(branches, expanded, expanded, key_padding_mask=pad, need_weights=False)[0]
            )
        hidden = branches[:, 0].reshape(b, candidates, -1)
        logits = self.score(hidden).squeeze(-1)
        valid = torch.arange(candidates, device=states.device)[None] < batch["counts"][:, None]
        mean = (hidden * valid[..., None]).sum(1) / batch["counts"][:, None]
        maximum = hidden.masked_fill(~valid[..., None], -torch.inf).max(1).values
        score_mean = (logits * valid).sum(1) / batch["counts"]
        score_max = logits.masked_fill(~valid, -torch.inf).max(1).values
        summary = torch.cat(
            (mean, maximum, score_mean[:, None], score_max[:, None], batch["counts"][:, None].float() / 7), dim=-1
        )
        none = self.none(summary).squeeze(-1)
        # 每題 none 放在自己的 count 位置，padding logits 保持 -inf。
        result = torch.full((b, candidates + 1), -torch.inf, device=states.device, dtype=logits.dtype)
        result[:, :candidates] = logits.masked_fill(~valid, -torch.inf)
        none = none.masked_fill(batch["kinds"] != TYPES["choice"], -torch.inf)
        result.scatter_(1, batch["counts"][:, None], none[:, None])
        return {"logits": result, "candidate_hidden": hidden, "memory": expanded}

    def memory_key(self, state):
        raw = text_bytes(state, 512, "state")
        digest = hashlib.sha256(raw)
        metadata = {
            "config": self.config.__dict__,
            "preprocessing": PREPROCESSING,
            "pad": PAD,
            "positions": "learned absolute, state starts0",
            "dtype": str(next(self.parameters()).dtype),
            "device": str(next(self.parameters()).device),
        }
        digest.update(json.dumps(metadata, sort_keys=True).encode())
        for name, tensor in self.state_encoder.state_dict().items():
            digest.update(name.encode())
            digest.update(tensor.detach().cpu().contiguous().view(torch.uint8).numpy().tobytes())
        return digest.hexdigest()

    @torch.no_grad()
    def state_memory(self, state):
        if self.training or next(self.parameters()).dtype != torch.float32:
            raise ValueError("快取只允許 frozen fp32 eval")
        key = self.memory_key(state)
        if key not in self._cache:
            device = next(self.parameters()).device
            tokens = pad_bytes([[CLS, *text_bytes(state, 512, "state")]], device)
            if len(self._cache) >= 64:
                self._cache.clear()
            self._cache[key] = StateMemory(key, self.encode_state(tokens), tokens.eq(PAD))
        cached = self._cache[key]
        # 呼叫者持有自己的副本；不能經公開 tensor 意外污染內部快取。
        return StateMemory(cached.key, cached.values.clone(), cached.padding.clone())

    @torch.no_grad()
    def predict(self, question, temperature=1.0, use_cache=True, memory=None):
        question.validate()
        if self.training or temperature <= 0:
            raise ValueError("推論須 eval 且 temperature > 0")
        if memory is not None and memory.key != self.memory_key(question.state):
            raise ValueError("state／encoder／tokenizer／dtype/device 已變，舊 cache 失效")
        if use_cache and memory is None:
            memory = self.state_memory(question.state)
        batch = collate_questions([question], next(self.parameters()).device)
        result = self(batch, memory=memory)
        ids = [candidate.id for candidate in question.candidates] + (["none"] if question.kind == "choice" else [])
        logits = result["logits"][0, : len(ids)]
        # raw logits 決平手，再按 stable ID；不讓 softmax 浮點捨入改答案。
        best = min(range(len(ids)), key=lambda i: (-float(logits[i]), ids[i]))
        probabilities = F.softmax(logits / temperature, dim=-1)
        return {
            "answer": ids[best],
            "ids": ids,
            "logits": logits.cpu().tolist(),
            "probabilities": probabilities.cpu().tolist(),
            "p_max": float(probabilities.max()),
            "candidate_hidden": result["candidate_hidden"][0].cpu(),
        }


# 完整正式架構的隨機初始化核對，沒有載入正式權重。
torch.set_num_threads(1)
torch.manual_seed(20261029)
formal_c = ChoiceModel().eval().requires_grad_(False)
example_state = "A: task=record; permission=no; ready=yes."
example_choice = Question(
    example_state,
    "choice",
    "Choose A action.",
    (Candidate("record", "Device A: record"), Candidate("stop", "Device A: stop")),
)
example_info = Question(
    example_state,
    "info",
    "Does A have all required fields?",
    (Candidate("yes", "Required information is sufficient"), Candidate("no", "Required information is missing")),
)
formal_memory = formal_c.state_memory(example_state)
saved_memory = formal_memory.values.clone()
fresh_prediction = formal_c.predict(example_choice, use_cache=False)
first_prediction = formal_c.predict(example_choice, memory=formal_memory)
formal_c.predict(example_info, memory=formal_memory)
last_prediction = formal_c.predict(example_choice, memory=formal_memory)
reverse_question = Question(example_state, "choice", example_choice.text, tuple(reversed(example_choice.candidates)))
reverse_prediction = formal_c.predict(reverse_question, memory=formal_memory)
fresh_error = max(abs(a - b) for a, b in zip(fresh_prediction["logits"], first_prediction["logits"]))
repeat_error = max(abs(a - b) for a, b in zip(first_prediction["logits"], last_prediction["logits"]))
p_before = dict(zip(first_prediction["ids"], first_prediction["probabilities"]))
p_after = dict(zip(reverse_prediction["ids"], reverse_prediction["probabilities"]))
print("正式架構width／heads", formal_c.config.width, formal_c.config.heads)
print("材料memory形狀", tuple(formal_memory.values.shape))
print("choice／info候選數", len(first_prediction["ids"]), len(formal_c.predict(example_info)["ids"]))
print("重算／共享及重問在1e-5內", fresh_error <= 1e-5, repeat_error <= 1e-5)
print("重排機率在1e-5內", max(abs(p_before[k] - p_after[k]) for k in p_before) <= 1e-5)
print("共用材料未被寫回", torch.equal(formal_memory.values, saved_memory))

核對印64維、4 heads，choice／info候選數3／2。材料memory是 `[1, 1+state的UTF-8 byte數, 64]`；補上的一格是CLS。三個誤差檢查與「未被寫回」應為True，並不要求浮點差恰好為0。這些是隨機模型的運算等價性，沒有宣稱它知道record或stop何者正確。

</details>

**正式資料究竟問什麼？** 前面的客服短例用來教接口；正式成績來自三個裝置A、B、C的有限規則世界。本段A／B／C都是裝置名稱；產品與路由段另用同樣字母作成品代號。每個裝置有想做的任務`task`、權限`permission`與就緒狀態`ready`；`yes/no`表示已知的是／否，`unknown`表示不知道，兩者不能混用。以下是固定final清單第一個家族的實際材料，不是另挑一題當成準確率：

```text
A: task=save; permission=no; ready=no. B: task=play; permission=yes; ready=no. C: task=record; permission=no; ready=no.
```

本世界把`record/play/save`用作錄音／播放／儲存，另外有`stop`停止、`wait`等待。Choice先要求A的permission與ready都已知；缺任一欄便以none表示無法按此契約選動作。兩欄已知時，先判permission：no就stop；否則ready為no就wait；其餘才做task所要求的動作。候選被移除也可使正解改為none，不能因材料完整就硬選剩下的動作。

這個家族的四筆實際英文問題如下；前三筆是三種主型代表題，第四筆另查Choice對象的必要資訊，**仍然是info型**：

```text
choice（kind=choice）
Choose A's action. Require permission and ready known; no permission: stop; not ready: wait; else requested task.

info（kind=info）
Does B have all required fields known? record needs permission+ready; play needs permission; save needs ready.

rubric（kind=rubric）
Grade C: record=count yes fields; play=2 if permission else ready(0/1); save=2 if ready else permission(0/1).

choice_info（kind=info）
Are permission and ready both known for A's action?
```

Choice候選ID依序為`save, wait, record, play, stop`，描述分別是`Device A: save`、`Device A: wait`、`Device A: record`、`Device A: play`、`Device A: stop`，模型最後另加none。兩個info問題都用`yes/no`，描述為`Required information is sufficient`／`Required information is missing`；rubric用`0/1/2`，描述為`Device C has grade 0/1/2`。各題依**自己的英文規則**求正解：

| 問題 | 本筆正解 | 由材料怎樣判 |
| --- | --- | --- |
| Choice問A應做什麼 | stop | A兩欄都已知，permission=no優先於ready=no |
| info問B的必要資訊是否齊備 | yes | B想play，只需permission已知；ready=no不等於資訊缺失 |
| rubric按規準替C給級 | 0 | C想record，規準數permission與ready中有幾個yes；這筆兩個都是no |
| choice_info查A能否作Choice判斷 | yes | A的permission與ready都已知；它不要求兩個值必須為yes |

本批正式rubric題的permission與ready都已知，0／1就是已知no／yes對應的數字；生成器拒絕用unknown建立rubric標籤。C-v2這個三分類介面未替缺欄另設none，因此不能由合法文字輸入推論它會正確拒答；[19.25](https://birdhackor.github.io/tiny-perceptron-vlm/19.25.html)的C-v3才把rubric缺欄的none納入服務與驗收。

這裡的rubric是事前寫明的裝置規則等級，不是通用作文或服務品質。接受Choice時，不能拿B的info代表題代替A的choice_info。再手工做兩個單一變化來核規則：只從上面候選刪掉stop，Choice正解變none、A的資訊仍充分；只把A的permission改成unknown，Choice同樣變none，但A的資訊變不充分。這兩題只教標註，不另加進已封存的final分母。

三個訓練表面格式是短欄位、英文句子與中文欄位。為看清寫法，先只固定上述A的已知事實，各寫一次（B／C也按同格式列出）：

```text
短欄位：A: task=save; permission=no; ready=no.
英文句子：Device A requests save. Its permission is no; its readiness is no.
中文欄位：裝置A：任務save；許可否；就緒否。
```

第四種`Status for A: requested action save, readiness no, permission no.`是留出的英文順序，正式訓練不用它，core final後30個家族才用。這些都是自製規則世界的寫法。實際訓練variant還按固定規則改問題措辭、候選順序及缺欄變體；上面固定事實的三行只說明寫法，不能把整個增廣比較當成只改文字格式、其餘可見欄位完全相同的因果實驗。

正式資料在同一個三裝置規則生成器中，排除實際可見內容重複後，使用903個train家族、64 dev、80 calibration、60 policy、60 core final，另外30個C整合家族放入Auto。主要模型每個train家族有3種表面格式、每種4個核心問題，共10836個核心訓練題；資料對照直接保留原variant0的3612題，不另外把這個對照組的缺欄補回。兩模型同架構、同初始化、同路由資料、各6000次更新；它比較上述多格式與缺欄／措辭變體合在一起的資料增廣，不單獨隔離某一種表面格式的收益，也不能把它當成舊C與新C的純架構比較。

兩個正式模型均已完成6000次更新，各279,170個參數，dev都選第5000步。先固定增廣版為本輪主要C，再由另外80個calibration家族選出T＝0.6484197773255048，60個policy家族選出接受門檻tau＝0.6；這兩個值在開final前已提交。門檻是在當時的policy資料有可行解，不能據此預先保證final的coverage或risk。

真正的60個核心final家族得到下面結果。每個家族都有Choice、資訊充分性及rubric問題，所以三列不是180個獨立家族。compact只作同架構的資料對照，沒有另外替它擬合T或接受門檻：

| 題型 | 增廣版，事前選定 | compact原始選答 | 增廣版的事前門檻 |
| --- | --- | --- | --- |
| Choice選候選 | 39／60＝65.0% | 28／60 | 至少54／60，未達 |
| 資訊是否充分 | 55／60＝91.7% | 42／60 | 至少51／60，達到 |
| rubric評價 | 35／60＝58.3% | 21／60 | 至少51／60，未達 |

Choice再分開看：完整資料36／48，移除正確候選後3／6，缺必要資料0／6。增廣版在本次對照比compact選對更多，仍沒有學成足以部署的判斷器；同一個初始化及一組資料對照，也不能證明增廣在所有seed或任務都較好。

固定T沒有改變39／60的最高候選，卻讓Choice平均負對數代價NLL由2.6901變為3.9624。這個代價按報給正確答案的機率計算，越小越好，見[9.18的完整計算](https://birdhackor.github.io/tiny-perceptron-vlm/9.18.html)。固定接受規則在60題中接受53題，裡面19題錯：coverage為88.3%，接受風險35.8%，遠高於事前5%的上限。原policy可行與final失準都保留，沒有看過這些答案後重新挑T或tau。

計算契約則有不同的結果：8個dev家族的共享／重算、Q1→Q2→Q1及重排核對全部在1e-5容差內通過。這證明本次檢查下快取與隔離的實作一致，不能代替上面的語義正確率。另將同60家族各改為2–8個候選（含none），七組正確數依序為37、22、29、37、37、32、26，各以60為分母；420筆是相關的候選數診斷，不增加獨立樣本數，也不是事後換候選來重報主要成績。

因此**本節C-v2只發布為研究候選，當時保留舊版C預設**；目前裝置決策改用[19.25](https://birdhackor.github.io/tiny-perceptron-vlm/19.25.html)另行驗收的C-v3。讀者可以用它檢查共享計算、候選語義及失敗，但不能把可輸入UTF-8或通過快取檢查當成可靠決策。逐題分數、候選數與所有分母在[本輪結果](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/omni-products/v1/results-summary.json)，下載與操作見[新版模型操作](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/omni-products/README.md)。

一個batch有一半核心問題、一半A／B／C／none路由題，路由四類等權。路由候選描述是任務入口；它不會看到圖片或聲音內容。核心final的48／6／6 panel仍按原先定義計數，三型相關不變。四種自製表面格式與英文／中文state轉寫都不等於獨立真實語料，也沒有證明成熟模型在預訓練時未見。
